# 4b - Debt capacity, built by hand

**Time:** 35 minutes  |  **AI: off** - you write every line yourself

**What you will do**
- Rebuild Session 0, slide 4: how much can a deal borrow?
- Write a **function of the unknown** (`dscr(debt)`), turn the covenant into a **gap function**, and give it to `brentq`.
- Check the answer three ways: put it back in, solve a simple case by hand, and break it with a downturn.

How to work: click a cell, press Shift+Enter to run it. Run the cells in order, top to bottom. Fill each blank, marked `YOUR CODE HERE`. Use `brentq` from 4a.

## The problem

A bank lends to a company. The bank sets a **covenant**: a rule in the loan contract. Here the rule is that the **debt service coverage ratio (DSCR)** must be at least 1.25x **in every year**. If it falls below, the bank can call the loan (demand its money back).

**DSCR** = cash available for debt / debt service (the repayment plus the interest). A DSCR of 1.25x means each $1 owed is covered by $1.25 of cash.

The question: **what is the most the company can borrow?**

| Input | Value |
|---|---|
| EBITDA today (year 0) | $20M, growing 5% a year |
| Cash available for debt service | 75% of EBITDA |
| Interest rate | 7%, paid on the balance at the start of the year |
| Repayment | 5 equal parts, one per year (years 1 to 5) |
| Covenant | DSCR of 1.25x or more, every year |

**EBITDA** is the company's yearly operating profit before interest, tax and depreciation. $20M means 20 million dollars. In Excel you would use Goal Seek. In Python you build the solver in Steps 1 to 4, then check it in Steps 5 to 8.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq   # the solver from 4a: `from ... import ...` takes one tool from a module

print("Imports loaded")

## Step 1 - The inputs: EBITDA, cash, rate, covenant

`np.arange(1, 6)` is the array `1, 2, 3, 4, 5`: the five years (it stops before the second number). Raising a number to an array gives one value per year, all at once. That is how you get the growth factor of each year.

- `ebitda`: the EBITDA of today, grown at 5% a year, for each of the five years. An array of five numbers (in $M).
- `cash`: the cash available for debt service, 75% of `ebitda`. Array times a number: every year at once.
- `rate` and `covenant`: the interest rate (7%) and the minimum DSCR (1.25), as decimals. Two names, so you can find and change them later. Both can go on one line.

You should get EBITDA **21.0, 22.05, 23.15, 24.31, 25.53**.

In [ ]:
# YOUR CODE HERE
# YOUR CODE HERE
# YOUR CODE HERE

print(ebitda.round(2))
print(cash.round(2))
print(rate, covenant)

## Step 2 - A function of the unknown: `dscr(debt)`

The unknown is the **debt**. For any debt you try, the function must return the five DSCRs, one per year. Write it as a **function of the unknown**: `def dscr(debt):`.

Inside the function:
1. The balance at the **start** of each year. Repaying in 5 equal parts means one fifth of the debt is repaid each year. Start of year 1: the whole debt. Start of year 2: the debt minus one repayment. Start of year 3: minus two repayments. And so on. `np.arange(5)` gives `0, 1, 2, 3, 4`: how many repayments were already made at the start of each year.
2. The debt service of each year: the repayment (one fifth of the debt) plus the interest (the balance times the rate).
3. The DSCR: `cash` divided by the debt service. Array divided by array: five ratios.

Use `return` to send back the five ratios.

In [ ]:
# YOUR CODE HERE

print(dscr(40).round(2))

**Check the function on one case.** Take a debt of 40. In year 1 you owe `40 / 5 = 8` of repayment and `40 * rate = 2.8` of interest: a debt service of 10.8. Your year-1 DSCR must be `cash[0] / 10.8`.

The guard stops the code with a message if your function does not agree.

In [ ]:
by_hand = cash[0] / (40 / 5 + 40 * rate)

if len(dscr(40)) != 5:
    raise ValueError("dscr must return five ratios, one per year.")
if not np.isclose(dscr(40)[0], by_hand):
    raise ValueError(f"Year-1 DSCR is {dscr(40)[0]}, the hand calculation gives {by_hand}.")
print("Check OK: dscr(40) agrees with the hand calculation for year 1")

## Step 3 - The gap function: `gap(debt)`

`brentq` finds where a function is **zero**. Your question is "the lowest DSCR is exactly the covenant", not "something is zero". So turn the question into a **gap function**: a function that is **zero at the answer**.

`gap(debt)` = the **worst year's DSCR** minus the covenant. Build it from `dscr`, `min` and `covenant`.

- `min` takes the smallest of the five ratios: the covenant must hold **every** year, so the worst year decides.
- Too much debt: the worst DSCR is below the covenant, the gap is **negative**.
- Little debt: the worst DSCR is above the covenant, the gap is **positive**.
- Zero: the largest debt the covenant allows.

In [ ]:
# YOUR CODE HERE

**Check the bracket.** `brentq(gap, 1, 500)` searches between debt 1 and debt 500. It only works if `gap` has **different signs** at the two ends (the error you saw in 4a, cell 8). A debt of 1 is safe, a debt of 500 is far too much. The guard writes the two signs in one line: `gap(1) > 0 > gap(500)` means `gap(1)` is above 0 **and** 0 is above `gap(500)`.

In [ ]:
if not (gap(1) > 0 > gap(500)):
    raise ValueError("gap must be positive at debt 1 and negative at debt 500: check your gap function.")
print("Check OK: the bracket 1 to 500 has a sign change, brentq can search it")

## Step 4 - Solve it: `brentq`

Give `brentq` the function `gap` without brackets (a **function as an input**), then the **bracket**: the search runs between $1M and $500M of debt. The answer is the debt where `gap` is zero. Save it in `max_debt`.

You should get **46.67** ($M).

In [ ]:
# YOUR CODE HERE

print(f"Maximum debt: {max_debt:.2f}")

## Step 5 - Put the answer back in

Two checks, as in 4a:
1. `gap(max_debt)` must be zero.
2. The DSCR by year at `max_debt`: the lowest must be exactly the covenant. **Year 1 is the tightest**: the balance is highest in year 1, so the interest is highest, while the cash is lowest.

You should get a DSCR of **1.25, 1.38, 1.54, 1.71, 1.92**.

In [ ]:
print("gap at the answer:", round(abs(gap(max_debt)), 10))
print("DSCR by year:", dscr(max_debt).round(2))

if abs(gap(max_debt)) > 1e-6:
    raise ValueError("The gap at max_debt is not zero.")
if not np.isclose(min(dscr(max_debt)), covenant):
    raise ValueError("The lowest DSCR must be equal to the covenant.")
print("Check OK: the answer put back in gives a lowest DSCR of exactly the covenant")

A picture: the DSCR by year at the maximum debt, with the covenant line. Run it, no blank.

In [ ]:
years = np.arange(1, 6)
ratios = dscr(max_debt)

plt.figure(figsize=(6, 3.4))
plt.bar(years, ratios, color="#0091D1")
plt.axhline(covenant, color="#C8102E", linestyle="--")
plt.text(5.4, covenant + 0.03, f"covenant {covenant}x", color="#C8102E", ha="right")
plt.xlabel("Year")
plt.ylabel("DSCR")
plt.title(f"Debt {max_debt:.1f}: the tightest year touches the covenant")
plt.show()

## Step 6 - Solve a simple version by hand

This is entry 15 of the checks card (linked at the bottom): **solve a simple version by hand**. Keep year 1 only. The covenant says:

cash of year 1 / (repayment + interest) = covenant

With a debt `D`: the repayment is one fifth of `D`, the interest is `D` times the rate, so the debt service is `D` times (one fifth plus the rate). Solve for `D`: it is the cash of year 1, divided by the covenant times (one fifth plus the rate). Save it in `hand_check`.

Year 1 is the tightest year, so this simple version must give the **same** number as the solver. You should get 15.75 / 0.3375.

In [ ]:
# YOUR CODE HERE

print(f"By hand: {hand_check:.2f}")

if not np.isclose(hand_check, max_debt):
    raise ValueError(f"By hand {hand_check:.4f}, solver {max_debt:.4f}: they must agree.")
print("Check OK: the hand calculation and the solver agree (Excel Goal Seek gives it too)")

## Step 7 - Change an assumption

The MD asks: "and if the bank asks for 1.5x? And at 9% interest?" You do not need new code. A function reads the **value of a variable at the moment it runs**. So change the input (`covenant`), run `brentq` again, and `gap` and `dscr` use the new value. First set the covenant to 1.5 and save the max debt in `debt_150`. Then the covenant at 1.0, in `debt_100`. Then the covenant back at 1.25 with the rate at 9%, in `debt_9pct`.

You should get a covenant of 1.5x: **38.89**. A covenant of 1.0x: **58.33**. An interest rate of 9% (with the covenant back at 1.25): **43.45**.

In [ ]:
# YOUR CODE HERE
# YOUR CODE HERE
# YOUR CODE HERE

print(f"Covenant 1.5x:   {debt_150:.2f}")
print(f"Covenant 1.0x:   {debt_100:.2f}")
print(f"Interest 9%:     {debt_9pct:.2f}")

**Is it plausible?** A stricter covenant means **less** debt, a looser one **more**. A higher interest rate means **less** debt. The guard checks the direction.

In [ ]:
if not (debt_150 < max_debt < debt_100):
    raise ValueError("A stricter covenant must allow less debt, a looser one more.")
if not (debt_9pct < max_debt):
    raise ValueError("A higher interest rate must allow less debt.")
print("Check OK: stricter covenant, less debt. Higher rate, less debt.")

**Put the inputs back.** `rate` is still 0.09 from the last test. Reset `rate` and `covenant` to the base case (the table at the top), so the next steps start from the original numbers. The guard solves again and must find the base answer.

In [ ]:
# YOUR CODE HERE

if not np.isclose(brentq(gap, 1, 500), max_debt):
    raise ValueError("The inputs are not back to the base case: max debt differs from max_debt.")
print("Check OK: back to the base case")

## Step 8 - The downturn: why `min`?

The plan above assumes EBITDA grows 5% every year. What if year 2 goes badly? EBITDA falls **25%** below plan in year 2: `22.05 * 0.75` is **16.54** instead of 22.05. The other years stay as planned.

- Make a copy of the array `ebitda` and call it `down_ebitda` (`copy`), so `ebitda` stays unchanged.
- Year 2 is position 1 of `down_ebitda`: Python counts from 0. Cut it by 25%.
- Then set `cash` to 75% of `down_ebitda`. `dscr` reads `cash` when it runs, so it now sees the downturn. Solve again with `brentq` and `gap`, with the same bracket as before, and save the answer in `max_debt_down`.

You should get a maximum debt of **38.76**: lower than before, and now **year 2** is the tightest.

In [ ]:
# YOUR CODE HERE
# YOUR CODE HERE
# YOUR CODE HERE

print("EBITDA with the downturn:", down_ebitda.round(2))
print(f"Max debt, downturn: {max_debt_down:.2f}")
print("DSCR by year:", dscr(max_debt_down).round(2))

Now a **wrong** solver: one that checks **year 1 only**. Its gap is the DSCR of the first year (position 0 of the array) minus the covenant. The downturn is in year 2, so it does not see it: it still says the old answer.

Write `gap_year1(debt)`, solve it with `brentq` and the same bracket, save the answer in `debt_year1`, and look at year 2. The cash still has the downturn in it.

In [ ]:
# YOUR CODE HERE
# YOUR CODE HERE

print(f"Year-1-only solver: {debt_year1:.2f}")
print("DSCR by year with the downturn:", dscr(debt_year1).round(2))

You should see the year-1-only solver still says **46.67**, and **year 2 breaks the covenant: DSCR 1.04**. The bank would call the loan. This is why `gap` uses `min()` over **all** years.

In [ ]:
safe = dscr(max_debt_down)
risky = dscr(debt_year1)
year2 = risky[1]

if not (debt_year1 > max_debt_down):
    raise ValueError("The year-1-only solver should allow more debt than the min solver.")
if year2 >= covenant:
    raise ValueError("Year 2 should break the covenant with the year-1-only debt.")
if not np.isclose(min(safe), covenant):
    raise ValueError("With min over all years, the lowest DSCR must be exactly the covenant.")
print(f"Check OK: year 2 is {year2:.2f}, below the covenant. min over all years protects the bank.")

A picture of the two debts under the downturn: the safe debt, and the year-1-only debt. Run it, no blank.

In [ ]:
x = np.arange(1, 6)

plt.figure(figsize=(6.4, 3.6))
plt.bar(x - 0.2, safe, 0.38, color="#0091D1", label=f"min over all years (debt {max_debt_down:.1f})")
plt.bar(x + 0.2, risky, 0.38, color=["#C8102E" if v < covenant - 1e-9 else "#B8BEC6" for v in risky],
        label=f"year 1 only (debt {debt_year1:.1f})")
plt.axhline(covenant, color="#C8102E", linestyle="--")
plt.xlabel("Year")
plt.ylabel("DSCR")
plt.title("Downturn in year 2: the year-1-only debt breaks the covenant")
plt.legend(frameon=False, fontsize=8, loc="upper left")
plt.ylim(0, 2.6)
plt.show()

**Put the cash back.** Set `cash` to the plan again (75% of `ebitda`, not `down_ebitda`). The guard solves again and must find the base answer.

In [ ]:
# YOUR CODE HERE

if not np.isclose(brentq(gap, 1, 500), max_debt):
    raise ValueError("The cash is not back to the plan: max debt differs from max_debt.")
print("Check OK: back to the plan")

## Stretch (if time) - The sponsor's return

A private equity **sponsor** (an investment firm) buys the company and uses the debt to pay part of the price. Does the debt raise the sponsor's return?

- Buy at **8x EBITDA today**: `8 * 20 = 160`.
- The sponsor pays `160 - debt` from its own pocket (the **equity**). Year 0: a negative cash flow.
- Each year, the cash for debt service pays the bank first (repayment plus interest); the **spare cash** goes to the sponsor.
- After 5 years the debt is fully repaid. The sponsor sells at **8x the year-5 EBITDA**: `8 * 25.53 = 204.2`.

`np.zeros(6)` makes six slots filled with zeros: year 0, then years 1 to 5 (Session 0, slide 10: one slot more than the years).

Write `sponsor_flows(debt)`. It returns `flows`, an array made with `np.zeros(6)`: year 0 is minus the equity (the price minus the debt). Years 1 to 5 are the spare cash: the cash minus the debt service (the repayment plus the interest on the balance, as in `dscr`). Year 5 also gets the sale at 8 times the year-5 EBITDA (the debt is repaid by then); `+=` adds to the number already there.

Then find the IRR with `brentq`, as in 4a, with a bracket from -50% to 100%: `npv_at(r)` reads the variable `flows`, so set `flows` first. `npv_at` is given. Save the IRR without debt in `irr_0`, and with the maximum debt in `irr_debt`.

You should get an IRR of **14.8%** with no debt, and **16.5%** with the maximum debt from Step 4.

In [ ]:
def npv_at(r):
    return sum(flows / (1 + r) ** np.arange(len(flows)))

# YOUR CODE HERE

flows_0 = sponsor_flows(0)
flows_debt = sponsor_flows(max_debt)
print("Flows, no debt:  ", flows_0.round(1))
print("Flows, with debt:", flows_debt.round(1))

In [ ]:
# YOUR CODE HERE
# YOUR CODE HERE

print(f"Sponsor IRR, no debt:       {irr_0:.1%}")
print(f"Sponsor IRR, with {max_debt:.1f} debt: {irr_debt:.1%}")

# Put the answer back in: the NPV at the IRR is zero
flows = flows_0
npv_0 = npv_at(irr_0)
flows = flows_debt
npv_d = npv_at(irr_debt)
if abs(npv_0) > 1e-6 or abs(npv_d) > 1e-6:
    raise ValueError("The NPV at the IRR must be zero.")
if not irr_debt > irr_0:
    raise ValueError("Cheap debt should raise the sponsor's return.")
print("Check OK: NPV at the IRR is zero, and the debt raises the return")

Leverage (buying with borrowed money) adds about 1.7 percentage points of return here: the debt costs 7%, the deal earns more than that. But the debt also leaves less room in a downturn (Step 8).

## What you learned

**Python**
- **Array of years**: `np.arange(1, 6)` is `1, 2, 3, 4, 5`. Arithmetic on an array works on every year at once.
- **Named inputs**: `rate, covenant = 0.07, 1.25` puts the assumptions in variables you can find and change.
- **Function of the unknown**: `def dscr(debt): ...` takes a debt and returns five ratios. `return` sends them back.
- **Gap function**: `min(dscr(debt)) - covenant` is zero exactly at the answer. Any Goal Seek target can be written this way.
- **Minimum**: `min(...)` takes the worst year, so the covenant holds in every year.
- **Solver with a bracket**: `brentq(gap, 1, 500)`; the gap must have different signs at the two ends.
- **A function reads a variable when it runs**: change `covenant`, `rate` or `cash`, run `brentq` again, and the new answer follows. Reset the inputs afterwards, and check you are back to the base.
- **Copy of an array**: `ebitda.copy()`, then change one position (`down_ebitda[1]`) without touching the original.

**Finance**
- DSCR = cash available / (repayment + interest). The covenant is a floor on it, every year.
- Maximum debt $46.67M at 1.25x. A stricter covenant (1.5x) gives less debt (38.89), a higher rate (9%) gives less (43.45).
- A downturn in year 2 cuts the maximum debt to 38.76. A solver that looks at year 1 only misses it.
- The recipe for any Goal Seek: write a function of the unknown, turn the target into a gap that is zero at the answer, give the solver the gap and a bracket, put the answer back in, solve a simple case by hand.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right